# Cost-effectiveness of AI-assisted CBME

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/hecon/blob/main/notebooks/02_cost_effectiveness_icer.ipynb)

The incremental cost-effectiveness ratio per additional competent learner, its uncertainty, an acceptability curve, one-way sensitivity and budget impact (paper, section 6). Every number is an assumption you set.

**How to use it.** Press ▶ on each step in order, or choose Runtime, then Run all. Each step's code is hidden behind its title; double-click the title to see it. Educational use only: nothing here is a medical device or a finding.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Code GPL-3.0-or-later; text CC BY-SA 4.0. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending). Cite with [CITATION.cff](https://github.com/zistgah/hecon/blob/main/CITATION.cff); the DOI is recorded in the repository README once minted.

In [ ]:
#@title Set up: load the hecon package from the repository { display-mode: "form" }
import subprocess, sys
REPO = "zistgah/hecon"  #@param {type:"string"}
try:
    import hecon
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/%s@main" % REPO], check=True)
    import hecon
print(hecon.NOTICE)

## Step 1: the ratio and its four quadrants

$$ICER = \frac{C_{AI} - C_{C}}{E_{AI} - E_{C}}$$

Here $E$ is the number of learners reaching the competency threshold. *Dominant* means more competent learners for less money; *dominated* means fewer or the same for more money; only the other two quadrants give a ratio worth reading.

In [ ]:
#@title Your assumptions { display-mode: "form" }
validity = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
ai_practice_mult = 1.6  #@param {type:"slider", min:1, max:3, step:0.1}
ai_feedback_mult = 1.5  #@param {type:"slider", min:1, max:3, step:0.1}
capture = 1.0  #@param {type:"slider", min:0, max:1, step:1}
cohort_size = 150  #@param {type:"integer"}
years = 10  #@param {type:"integer"}
tech_fixed = 20000  #@param {type:"number"}
governance_fixed = 15000  #@param {type:"number"}
cultivation_cost = 12000  #@param {type:"number"}
from hecon import model
P = dict(validity=validity, ai_practice_mult=ai_practice_mult, ai_feedback_mult=ai_feedback_mult, capture=capture,
         cohort_size=cohort_size, years=years, tech_fixed=tech_fixed, governance_fixed=governance_fixed, cultivation_cost=cultivation_cost)
R = model.simulate(P)
print(model.report(R))
for arm in ("harvest", "cultivate"):
    ic = R["icer"][arm][-1]
    print(arm, "vs comparator, cumulative:", ("%.0f per additional competent learner" % ic["value"]) if ic["label"] == "ratio" else ic["label"])

## Step 2: uncertainty and the acceptability curve

The probabilistic analysis draws the uncertain parameters from ranges around your values (the ranges are listed in `hecon.economics.UNCERTAIN`). The curve shows, for each amount you would pay per additional competent learner, the share of draws in which cultivation is worth it.

In [ ]:
#@title Probabilistic sensitivity analysis { display-mode: "form" }
from hecon import economics
import matplotlib.pyplot as plt
draws = 300  #@param {type:"integer"}
S = economics.psa(P, n=draws, arm="cultivate")
W = [0, 1000, 2000, 3000, 4000, 6000, 8000, 12000, 16000]
C = economics.ceac(S, W)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter([de for dc, de in S], [dc for dc, de in S], s=8); ax[0].axhline(0, color="grey", lw=0.8); ax[0].axvline(0, color="grey", lw=0.8)
ax[0].set_xlabel("extra competent learners"); ax[0].set_ylabel("extra cost"); ax[0].set_title("Cost-effectiveness plane")
ax[1].plot([w for w, _ in C], [p for _, p in C], marker="o"); ax[1].set_ylim(0, 1)
ax[1].set_xlabel("willingness to pay per additional competent learner"); ax[1].set_title("Acceptability curve"); plt.tight_layout(); plt.show()

## Step 3: which assumption decides the answer

One-way sensitivity moves one parameter at a time to 75 and 125 per cent of your value. The longest bar is the assumption to measure first.

In [ ]:
#@title One-way sensitivity { display-mode: "form" }
ref, rows = economics.one_way(P, arm="cultivate")
rows = [r for r in rows if r[1] is not None and r[2] is not None]
rows.sort(key=lambda r: abs(r[2] - r[1]))
plt.figure(figsize=(7, 4))
for i, (k, lo, hi) in enumerate(rows):
    plt.plot([lo, hi], [i, i], lw=6)
plt.yticks(range(len(rows)), [r[0] for r in rows]); plt.xlabel("ICER at 75% and 125% of the value"); plt.title("Tornado"); plt.show()

## Step 4: budget impact

The extra cost each year compared with conventional teaching, which is what an institution has to find.

In [ ]:
#@title Budget impact { display-mode: "form" }
bi = economics.budget_impact(R, "cultivate")
plt.figure(figsize=(7, 3.5)); plt.bar(R["years"], bi); plt.axhline(0, color="grey", lw=0.8)
plt.xlabel("year"); plt.title("Extra cost per year, cultivate vs comparator"); plt.show()

## Knowledge is not competence

The pooled evidence the paper cites measures knowledge acquisition, not clinical reasoning: Wei and colleagues (2025) pooled four randomised trials for knowledge (SMD 0.46, 95% CI 0.18 to 0.73, P = .001) and could not pool clinical reasoning because of methodological and measurement heterogeneity. *Evidence state: verified on 5 November 2026 against the full text in PubMed Central (PMC12268559).* Do not enter a knowledge effect as if it were a competence effect. Report any completed evaluation with CHEERS 2022.